# Labeling the Dataset using the trained student

What we do:
1) Run the student model on the dataset of inputs
2) Analyse the student dataset and label it


Student: Qwen2.5-1.5B-Instruct

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [16]:
from core.types import *
from core.utils.huggingface_client import HuggingFaceClient
from core.utils.huggingface_inference_client import HuggingFaceInferenceClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from sklearn.cluster import DBSCAN
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from ollama import ChatResponse
from openai.types.responses import Response as OpenAIResponse
from transformers import AutoTokenizer

import os
import json
import numpy as np
import pandas as pd

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
inputs = LLMCommandingInput.load_inputs(
    path=Path("data/inputs/inputs.json"),
    gstype=DoomGameState
)

inputs_lookup = {inp.id: inp for inp in inputs}

In [5]:
# For now, only extract inputs to label
selected_inputs = [
    inp
    for inp in inputs
    if inp.selected_for_labelling
]

print(f"Selected inputs: {len(selected_inputs)}/{len(inputs)}")

Selected inputs: 50/2872


In [14]:
student_client = HuggingFaceInferenceClient[LLMCommandingInput, LLMCommandingOutput](
    #model = "Qwen/Qwen2.5-1.5B-Instruct",
    model = "./models/training/final",
    max_output_tokens=512,
    temperature=0.0,
    working_dir=Path('data/huggingface'),
    use_flash_attention_2=False,
    load_in_4bit=True
)

🚀 Initialized HuggingFaceInferenceClient for ./models/training/final
   Device: cuda
   Flash Attention 2: False
   Quantization: 4-bit


In [7]:
# Prepare Prompt (same as training)
system_prompt = "You are a game command parser that converts natural language commands into DSL instructions."

In [8]:
def format_input(inp: LLMCommandingInput) -> str:
    game_state = inp.game_state.state.to_prompt_ready()
    command = inp.user_command.command.command
    return f"Game State:\n{game_state}\nCommand:\n{command}"


def parse_output(response: str, input_id: str, latency: float) -> LLMCommandingOutput:
    return LLMCommandingOutput(
        input_id=input_id,
        actions=response,
        reason=None,
        latency=latency,
    )


def get_id(gse: LLMCommandingInput, idx: int) -> str:
    return gse.id

In [9]:
print(system_prompt)
print(format_input(inputs[3]))

You are a game command parser that converts natural language commands into DSL instructions.
Game State:
AIMED_AT:
  type: Wall
  distance: 330.86
  interactable: yes

MONSTERS (count=0):

INVENTORY:
  current_slot: 2
  weapons:
    - (1, Fist, 0)
    - (2, Pistol, 50)
Command:
Go press that switch ahead


In [19]:
# Load tokenizer from BASE MODEL (not checkpoint)
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",  # ← Use original model
    trust_remote_code=True,
)

student_client.tokenizer = tokenizer

In [21]:
outputs = student_client.process(
    dataset=selected_inputs,
    system_prompt=system_prompt,
    tools = [], # No tools at level 3
    format_input=format_input,
    parse_output=parse_output,
    get_id=get_id,
    # batch_size=200,
)


📦 Loading model: ./models/training/final
   📉 Using 4-bit quantization (NF4)
   ✓ Model loaded successfully!

🔄 Processing 50 items sequentially


Processing items:   0%|          | 0/50 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [1] Message building: 0.000s
  [2] Chat template: 0.005s
  [2a] Prompt length: 369 chars
  [3] Tokenization: 0.004s
  [3a] Input tokens: 98


Processing items:   2%|▏         | 1/50 [00:01<00:54,  1.11s/it]

  [4] Generation: 1.079s
  [4a] Output tokens: 116
  [5] Decoding: 0.018s
  [5a] Response length: 36 chars
  [TOTAL]: 1.107s

LLMCommandingOutput(input_id='state-233-p2-uc0', actions='SPRINT 0.0 330.86\nINTERACT<|im_end|>', reason=None, latency=1.1061294078826904)
  [1] Message building: 0.000s
  [2] Chat template: 0.002s
  [2a] Prompt length: 408 chars
  [3] Tokenization: 0.000s
  [3a] Input tokens: 118


...

  [4] Generation: 0.885s
  [4a] Output tokens: 351
  [5] Decoding: 0.000s
  [5a] Response length: 49 chars
  [TOTAL]: 0.885s

LLMCommandingOutput(input_id='state-57139-p0-uc0', actions='ROTATE_TO_TARGET MONSTER_2\nFIRE_SHOTS 1<|im_end|>', reason=None, latency=0.8849365711212158)
  [1] Message building: 0.000s
  [2] Chat template: 0.000s
  [2a] Prompt length: 872 chars
  [3] Tokenization: 0.000s
  [3a] Input tokens: 453


Processing items:  98%|█████████▊| 49/50 [00:42<00:00,  1.09it/s]

  [4] Generation: 0.963s
  [4a] Output tokens: 469
  [5] Decoding: 0.000s
  [5a] Response length: 49 chars
  [TOTAL]: 0.964s

LLMCommandingOutput(input_id='state-57195-p3-uc0', actions='ROTATE_TO_TARGET MONSTER_3\nFIRE_SHOTS 5<|im_end|>', reason=None, latency=0.9639425277709961)
  [1] Message building: 0.000s
  [2] Chat template: 0.000s
  [2a] Prompt length: 447 chars
  [3] Tokenization: 0.001s
  [3a] Input tokens: 146


Processing items: 100%|██████████| 50/50 [00:43<00:00,  1.15it/s]

  [4] Generation: 0.480s
  [4a] Output tokens: 155
  [5] Decoding: 0.001s
  [5a] Response length: 36 chars
  [TOTAL]: 0.482s

LLMCommandingOutput(input_id='state-57954-p1-uc1', actions='ROTATE_TO_TARGET MONSTER_0<|im_end|>', reason=None, latency=0.4818427562713623)

✅ Completed: 50/50 successful


In [22]:
# Clustering was already made earlier, so inputs are already partitioned.
# Now, considering this is just a test of the Teacher's quality (to save time):
# - Having retrieved the LLMCommandingOutputs, I can just prepare the csv
# - This time, every row in the csv should be set to be evaluated.
# For the full run: check if selected. (MAKE SURE TO CHANGE FILE NAME SO THAT I DO NOT HAVE TO REVALUATE IF THEY ALREADY CORRECT)

rows = []
for idx, output in enumerate(outputs):
    inp = inputs_lookup[output.input_id]

    row = LLMCommandingLabelledDataPoint(
        input_id=inp.id,
        game_state=inp.game_state.state.to_prompt_ready(),
        command=inp.user_command.command.command,
        command_intent=inp.user_command.command.intent,
        command_explicitness=inp.user_command.command.explicitness,
        command_atomicity=float(inp.user_command.command.atomicity),
        command_contextuality=float(inp.user_command.command.contextuality),
        game_actions=output.actions.__str__(),
        latency=output.latency,
        reason_if_failed=output.reason,
        cluster_id=inp.cluster_id,
        selected_for_labelling=inp.selected_for_labelling,
    )

    rows.append(asdict(row))

df = pd.DataFrame(rows)

In [23]:
output_path = Path("data/outputs/selected-data-training-qwen-1.5b.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)